# Model Tuning

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import optuna

DB_FILENAME = "../data/model_tuning.db" 
STORAGE_URL = f"sqlite:///{DB_FILENAME}"



optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/liuhaochen/Projects/dota2pred/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

Models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000  # Increase if convergence issues
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        n_estimators=100
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=42,
        objective='binary',
        verbosity=-1,  # Suppress output
        n_estimators=100
    )
}

In [4]:
from pathlib import Path
import pandas as pd

file_path = Path("../data/model_tuning").resolve()
parquet_files = ["features_train.parquet", "features_test.parquet", "y_train.parquet", "y_test.parquet"]

def read_parquet_safe(base: Path, name: str) -> pd.DataFrame:
    p = base / name
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")
    return pd.read_parquet(p)  # engine auto-detected; works with zstd/gzip, etc.

X_train = read_parquet_safe(file_path, "features_train.parquet")
X_test  = read_parquet_safe(file_path, "features_test.parquet")
y_train = read_parquet_safe(file_path, "y_train.parquet")
y_test  = read_parquet_safe(file_path, "y_test.parquet")

print("Loaded:",
      X_train.shape, X_test.shape, y_train.shape, y_test.shape)





Loaded: (32433, 7) (8109, 7) (32433, 2) (8109, 2)


In [5]:
from lib.hyperparams.model_tuners import ModelTuner

In [7]:
logistic_tuner = ModelTuner(
    model_name="logistic_regression",
    X_train=X_train,
    y_train=y_train,
    target_column='radiant_win',
    evaluation_metric='accuracy',
)

In [8]:
best_params = logistic_tuner.tune(n_trials=50, study_name="logistic_regression_tuning")



--- Starting Hyperparameter Tuning for LOGISTIC_REGRESSION ---
Optimizing for: ACCURACY
Number of trials: 50


/Users/liuhaochen/Projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/liuhaochen/Projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/liuhaochen/Projects/dota2pred/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(



--- Tuning Complete ---
Best CV accuracy: 0.5705
Best parameters found: {'C': 9613.457373002877, 'penalty': 'l1'}


In [9]:
logistic_tuner.validate_on_test_set(X_test, y_test)


--- Validating Best LOGISTIC_REGRESSION Model on Test Set ---
Using parameters: {'C': 9613.457373002877, 'penalty': 'l1'}

Test Set Performance:
  - Accuracy: 0.5731
  - Log Loss: 0.6764
  - Roc Auc: 0.5981


{'accuracy': 0.5730669626341103,
 'log_loss': 0.6763664274925478,
 'roc_auc': 0.5981277655577333}

In [10]:
lgb_tuner = ModelTuner(
    model_name="lightgbm",
    X_train=X_train,
    y_train=y_train,
    target_column='radiant_win',
    evaluation_metric='accuracy',
)

In [11]:
lgb_tuner.tune(n_trials=50, study_name="lightgbm_tuning")


--- Starting Hyperparameter Tuning for LIGHTGBM ---
Optimizing for: ACCURACY
Number of trials: 50

--- Tuning Complete ---
Best CV accuracy: 0.5672
Best parameters found: {'learning_rate': 0.0028446709168278945, 'n_estimators': 300, 'num_leaves': 419, 'max_depth': 4, 'reg_alpha': 0.0013490186254920158, 'reg_lambda': 8.790677385095377}


{'learning_rate': 0.0028446709168278945,
 'n_estimators': 300,
 'num_leaves': 419,
 'max_depth': 4,
 'reg_alpha': 0.0013490186254920158,
 'reg_lambda': 8.790677385095377}

In [12]:
lgb_tuner.validate_on_test_set(X_test, y_test)


--- Validating Best LIGHTGBM Model on Test Set ---
Using parameters: {'learning_rate': 0.0028446709168278945, 'n_estimators': 300, 'num_leaves': 419, 'max_depth': 4, 'reg_alpha': 0.0013490186254920158, 'reg_lambda': 8.790677385095377}

Test Set Performance:
  - Accuracy: 0.5761
  - Log Loss: 0.6798
  - Roc Auc: 0.5981


{'accuracy': 0.5761499568380811,
 'log_loss': 0.6798251147413978,
 'roc_auc': 0.5981182591918305}

In [6]:
xgboost_tuner = ModelTuner(
    model_name="xgboost",
    X_train=X_train,
    y_train=y_train,
    target_column='radiant_win',
    evaluation_metric='accuracy',
)
xgboost_tuner.tune(n_trials=50, study_name="xgboost_tuning")



--- Starting Hyperparameter Tuning for XGBOOST ---
Optimizing for: ACCURACY
Number of trials: 50

--- Tuning Complete ---
Best CV accuracy: 0.5695
Best parameters found: {'learning_rate': 0.0020808073027228135, 'n_estimators': 1800, 'max_depth': 3, 'lambda': 0.5637668331024149, 'alpha': 4.4559663999489804e-08}


{'learning_rate': 0.0020808073027228135,
 'n_estimators': 1800,
 'max_depth': 3,
 'lambda': 0.5637668331024149,
 'alpha': 4.4559663999489804e-08}

In [7]:
xgboost_tuner.validate_on_test_set(X_test, y_test)


--- Validating Best XGBOOST Model on Test Set ---
Using parameters: {'learning_rate': 0.0020808073027228135, 'n_estimators': 1800, 'max_depth': 3, 'lambda': 0.5637668331024149, 'alpha': 4.4559663999489804e-08}

Test Set Performance:
  - Accuracy: 0.5749
  - Log Loss: 0.6745
  - Roc Auc: 0.6011


{'accuracy': 0.5749167591564928,
 'log_loss': 0.6745183007669292,
 'roc_auc': 0.6010532197541671}

In [8]:
rf_tuner = ModelTuner(
    model_name="random_forest",
    X_train=X_train,
    y_train=y_train,
    target_column='radiant_win',
    evaluation_metric='accuracy',
)
rf_tuner.tune(n_trials=50, study_name="random_forest_tuning")



--- Starting Hyperparameter Tuning for RANDOM_FOREST ---
Optimizing for: ACCURACY
Number of trials: 50

--- Tuning Complete ---
Best CV accuracy: 0.5699
Best parameters found: {'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 14, 'min_samples_leaf': 13, 'criterion': 'gini'}


{'n_estimators': 200,
 'max_depth': 5,
 'min_samples_split': 14,
 'min_samples_leaf': 13,
 'criterion': 'gini'}

In [9]:
rf_tuner.validate_on_test_set(X_test, y_test)


--- Validating Best RANDOM_FOREST Model on Test Set ---
Using parameters: {'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 14, 'min_samples_leaf': 13, 'criterion': 'gini'}

Test Set Performance:
  - Accuracy: 0.5784
  - Log Loss: 0.6753
  - Roc Auc: 0.6014


{'accuracy': 0.5783697126649402,
 'log_loss': 0.6753287427364795,
 'roc_auc': 0.6014180074412532}